# Train Baseline & Advanced Models

Notebook này huấn luyện các mô hình baseline và mô hình nâng cao để phục vụ đánh giá:

- Baseline: Logistic Regression, LinearSVC, Multinomial Naive Bayes.
- Advanced: Random Forest, LightGBM nếu môi trường đã cài `lightgbm`.
- Vectorizer: TF-IDF với `ngram_range=(1, 3)`, `min_df=2`.
- Output: dự đoán test set trong `results/predictions/` và model trong `models/baselines/`.

In [14]:
from pathlib import Path
import json
import warnings

import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [15]:
DATA_PATH = Path("../Dataset/processed/cti_dataset_cleaned.csv")
PREDICTIONS_DIR = Path("../results/predictions")
MODELS_DIR = Path("../models/baselines")

LABEL_COL = "label"
TEXT_COLUMN_PRIORITY = ["text_nlp", "text_clean", "text"]

TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_FEATURES = 50000
N_ESTIMATORS = 300

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
df = pd.read_csv(DATA_PATH)

available_text_cols = [col for col in TEXT_COLUMN_PRIORITY if col in df.columns]
if not available_text_cols:
    raise ValueError(f"Không tìm thấy cột văn bản. Cần một trong: {TEXT_COLUMN_PRIORITY}")
if LABEL_COL not in df.columns:
    raise ValueError(f"Không tìm thấy cột nhãn: {LABEL_COL}")

TEXT_COL = available_text_cols[0]
work_df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
work_df[TEXT_COL] = work_df[TEXT_COL].astype(str).str.strip()
work_df[LABEL_COL] = work_df[LABEL_COL].astype(str).str.strip()
work_df = work_df[(work_df[TEXT_COL] != "") & (work_df[LABEL_COL] != "")]

print(f"Dataset gốc: {df.shape}")
print(f"Dataset dùng train/evaluate: {work_df.shape}")
print(f"Cột text đang dùng: {TEXT_COL}")
print(f"Số label: {work_df[LABEL_COL].nunique()}")
work_df.head()

Dataset gốc: (13161, 17)
Dataset dùng train/evaluate: (13161, 2)
Cột text đang dùng: text_clean
Số label: 154


,text_clean,label
0,Authentication Bypass via SQL Injection. A log...,T1078
1,Union-Based SQL Injection. This attack occurs ...,T1190
2,Error-Based SQL Injection. This attack occurs ...,T1190
3,"Blind SQL Injection. In Blind SQL Injection, t...",T1190
4,Second-Order SQL Injection. In a Second-Order ...,T1505


In [17]:
X = work_df[TEXT_COL]
y = work_df[LABEL_COL]
stratify = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify,
)

print(f"Train: {len(X_train):,}")
print(f"Test: {len(X_test):,}")

Train: 10,528
Test: 2,633


In [18]:
def make_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=2,
        max_features=MAX_FEATURES,
        sublinear_tf=True,
    )


pipelines = {
    "logistic_regression": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
    "linear_svc": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "multinomial_nb": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", MultinomialNB()),
    ]),
    "random_forest": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}
pipelines["lightgbm"] = Pipeline([
    ("tfidf", make_tfidf()),
    ("classifier", LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        objective="multiclass",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )),
])
list(pipelines.keys())

['logistic_regression',
 'linear_svc',
 'multinomial_nb',
 'random_forest',
 'lightgbm']

## 4. Train models and save predictions

In [19]:
metadata = {
    "data_path": str(DATA_PATH),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "max_features": MAX_FEATURES,
    "n_estimators": N_ESTIMATORS,
    "models": list(pipelines.keys()),
}

with open(PREDICTIONS_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

for model_name, pipeline in pipelines.items():
    print(f"\n[*] Training {model_name}...")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    prediction_df = pd.DataFrame({
        "text": X_test.reset_index(drop=True),
        "true_label": y_test.reset_index(drop=True),
        "predicted_label": pd.Series(y_pred),
    })
    prediction_df.to_csv(PREDICTIONS_DIR / f"{model_name}_predictions.csv", index=False)
    joblib.dump(pipeline, MODELS_DIR / f"{model_name}.joblib")
    print(f"[+] Saved predictions and model for {model_name}")


[*] Training logistic_regression...
[+] Saved predictions and model for logistic_regression

[*] Training linear_svc...
[+] Saved predictions and model for linear_svc

[*] Training multinomial_nb...
[+] Saved predictions and model for multinomial_nb

[*] Training random_forest...
[+] Saved predictions and model for random_forest

[*] Training lightgbm...
[+] Saved predictions and model for lightgbm
